# 12주차 실습 — 원무 데이터로 보는 병원 운영

- **데이터**: 가상 병원 CDW (2024년 · 병동 6곳). 실제 병원 자료가 아닌 수업용 가상 자료입니다.
- **오늘 배우는 것**: 입원 · 퇴원 세기, 재원일수, 병상 가동률, 진료비와 청구

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "hospital_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/hospital_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 1MB) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNqVk8Fr2lAcx/+VR04tjGIyY8xgt10GZdDDTmOHaMwmSB1a2KEUYhe3tDqqQzHSRCy1k4GHbM0wDtml/0mPeb/8D3svSd2LcwdPefnmvV++n3y/OeYOeO4JOubq5TfkysmSrKmyKIlyVuQeIa76H71WfV8nOk+WxWqFLl9x4dCCUQdMi3tN5LflwyN6MlbxzRIFro6/zsLTCT7vIRh4MGrilolbkz1ElnDZef4M4SsbgeHjb16ioeC7R0aSswifWmD9TI5E5mpqqVZSyVs0pVIvUeXdUbl6qFQepBOiHQgsoKSoopbX5GKOz7OA6/pGwE8ecRQbSjHGVsFZMvbvdeuBwHdhot/rQwY4HhX2bUI1hqkdfQ/HQNBrbA/4mAVUc0pREDVJkbMpwHV9EyCMZpRj0k3R4YVJsvsCjo6eohcv9/fjlNzfkWHTIt4/wnkSDNqRArePaBGulyjsW1HYrQndx+9uQ5VlqZR8rkASEnJiocBSresJlZSimhr4uh3cLumG8OIsmI8TUGf5T1ujXOiDQQcMOynh35xoj/Gvz9tnJLI0gqyJoixlRFGQWJp1fVNG9C+6GOKbccr3SiURsXDobo528G0XPjSiqO4Gq8KZ1i695TOZPSRkhCxuGjRiAS574HhIkMmEYNGGqU4mmquAt4HOsdBaUZMFTeUlLaOx0Ov6Jmi8MAL/DAazwDehf5X++WzSwqaOgh8uyRgcKy4rEUddAoR7Llhe2LVpaQO3Ay0fBXMft/WoxePG1mQnfwDUbjFG")   # 자동 확인용 (정답 SQL 은 없음)

## 데이터 구성 — 가상 병원 CDW

실제 병원 자료가 아니라 수업용으로 만든 가상 자료입니다 (2024년 · 병동 6곳). 열 이름이 생각나지 않으면 이 표로 돌아오세요.

| 표 | 한 행이 뜻하는 것 | 기본키 (PK) | 외래키 (FK) | 주로 쓰는 열 |
|---|---|---|---|---|
| `환자` | 환자 1명 | `환자ID` | — | 성별, 생년월일, 사망일 |
| `입원` | 입원 1건 | `입원ID` | `환자ID` → 환자, `병동` → 병동 | 병동, 진료과, 입원일, 퇴원일, 퇴원형태, 계획입원, 보험유형 |
| `진단` | 진단 1건 | `진단ID` | `입원ID` → 입원 | 진단코드(KCD), 주진단여부 |
| `검사` | 검사 1건 | `검사ID` | `환자ID` → 환자 | 검사일, 검사명, 결과값 |
| `병동` | 병동 1곳 | `병동` | — | 구분, 주진료과, 병상수 |
| `병동일보` | 병동 × 날짜 | `병동 + 날짜` | `병동` → 병동 | 재원환자수(자정 기준), 입원환자수, 퇴원환자수 |
| `근무` | 병동 × 날짜 × 근무조 | `병동 + 근무일 + 근무조` | `병동` → 병동 | 근무조(D · E · N), 간호사수 |
| `진료비` | 입원 × 항목 | `진료비ID` | `입원ID` → 입원 | 항목, 금액, 급여구분 |
| `낙상` | 낙상 1건 | `낙상ID` | `입원ID` → 입원 | 발생일, 근무조, 손상정도 |

- 환자 1명이 여러 번 입원하면 `입원` 표에 여러 행이 있습니다 (1 : N). 입원 1건에 진료비 · 진단 · 낙상도 여러 행일 수 있습니다.
- 표를 이을 때는 `ON 낙상.입원ID = 입원.입원ID` 처럼 외래키와 기본키를 맞춥니다.
- 2024-12-31 에 아직 재원 중인 입원은 `퇴원일` 이 비어 있습니다(NULL). 환자ID 는 앞자리 0 이 있는 글자입니다.

## 1. 원무 업무와 데이터

입원 표의 한 행 = 입원 1건. 같은 환자가 여러 번 입원하면 여러 행입니다.

In [ ]:
%%sql
-- [2번] 환자 0139420 의 입원 3건 — 마지막 입원은 퇴원일이 NULL (재원 중)
SELECT 입원ID, 병동, 입원일, 퇴원일
FROM 입원
WHERE 환자ID = '0139420'
ORDER BY 입원일;

## 2. 입원과 퇴원 세기

In [ ]:
%%sql
-- [3번] 입원 10,030건 · 환자 7,970명 — 입원 건수 ≠ 환자 수
SELECT COUNT(*) AS 입원건수,
       COUNT(DISTINCT 환자ID) AS 환자수
FROM 입원;

In [ ]:
%%sql
-- [4번] ‘2024년’ 세 가지 기준 → 입원 9,840 · 퇴원 9,825 · 재원 10,030
SELECT COUNT(CASE WHEN YEAR(입원일) = 2024
                  THEN 1 END) AS 입원기준,
       COUNT(CASE WHEN YEAR(퇴원일) = 2024
                  THEN 1 END) AS 퇴원기준,
       COUNT(*) AS 재원기준
FROM 입원;

In [ ]:
%%sql
-- [5번] 12월 31일 재원 중 205명 — = NULL 이 아니라 IS NULL
SELECT 병동, COUNT(*) AS 재원중
FROM 입원
WHERE 퇴원일 IS NULL
GROUP BY 병동
ORDER BY 병동;

## 3. 재원일수

재원일수 = 퇴원일 − 입원일 (날짜끼리 빼면 일 수, 당일 입퇴원은 0일)

In [ ]:
%%sql
-- [6번] 입원마다 재원일수 — 재원 중이면 NULL
SELECT 입원ID, 입원일, 퇴원일,
       퇴원일 - 입원일 AS 재원일수
FROM 입원
WHERE 환자ID = '0139420'
ORDER BY 입원일;

In [ ]:
%%sql
-- [7번] 병동별 평균 재원일수 (전체 7.2일)
SELECT 병동, COUNT(*) AS 퇴원건수,
       ROUND(AVG(퇴원일 - 입원일), 1) AS 평균재원일수
FROM 입원
WHERE 퇴원일 BETWEEN '2024-01-01' AND '2024-12-31'
GROUP BY 병동
ORDER BY 병동;

In [ ]:
%%sql
-- [8번] 가장 긴 재원 65일 — 평균 옆에 최댓값도 봅니다
SELECT 입원ID, 병동, 퇴원일 - 입원일 AS 재원일수
FROM 입원
WHERE 퇴원일 IS NOT NULL
ORDER BY 재원일수 DESC, 입원ID
LIMIT 5;

## 4. 병상 가동률

가동률(%) = 재원일수 ÷ (병상 수 × 날짜 수) × 100. 2024년은 366일입니다.

In [ ]:
%%sql
-- [9번] 병동일보 — 한 행 = 병동 × 날짜, 재원환자수 = 그날 자정의 재원 환자
SELECT 날짜, 재원환자수,
       입원환자수 AS 입원, 퇴원환자수 AS 퇴원
FROM 병동일보
WHERE 병동 = '51병동'
ORDER BY 날짜
LIMIT 3;

In [ ]:
%%sql
-- [10번] 병동별 가동률 (전체 85.0%)
SELECT 병동일보.병동, 병상수,
       SUM(재원환자수) AS 재원일수,
       ROUND(100.0 * SUM(재원환자수)
             / (병상수 * 366), 1) AS 가동률
FROM 병동일보
JOIN 병동 ON 병동일보.병동 = 병동.병동
GROUP BY 병동일보.병동, 병상수
ORDER BY 병동일보.병동;

## 5. 진료비와 청구

In [ ]:
%%sql
-- [11번] 입원 1건(입원ID 240309)의 진료비 7행 — 입원 1건 : 진료비 여러 행
SELECT 항목, 금액, 급여구분
FROM 진료비
WHERE 입원ID = 240309
ORDER BY 진료비ID;

In [ ]:
%%sql
-- [12번] 진료과별 입원 1건당 진료비 (만 원) — 건수는 COUNT(DISTINCT 입원ID)
SELECT 진료과,
       COUNT(DISTINCT 진료비.입원ID) AS 입원건수,
       ROUND(SUM(금액) / COUNT(DISTINCT 진료비.입원ID)
             / 10000) AS 건당진료비
FROM 진료비
JOIN 입원 ON 진료비.입원ID = 입원.입원ID
GROUP BY 진료과
ORDER BY 건당진료비 DESC;

In [ ]:
%%sql
-- [13번] 급여 · 비급여 비율 — 청구 자료에는 급여만
SELECT 급여구분, SUM(금액) AS 금액,
       ROUND(100.0 * SUM(금액)
             / (SELECT SUM(금액) FROM 진료비), 1) AS 비율
FROM 진료비
GROUP BY 급여구분
ORDER BY 급여구분;

**JOIN 함정** — 진료비를 붙인 표로 재원일수를 더하면 몇 배로 커집니다.

In [ ]:
%%sql
-- [14번] 이렇게 하면 → 433,629 (진료비 행마다 재원일수가 다시 더해짐)
SELECT SUM(퇴원일 - 입원일) AS 재원일수
FROM 입원
JOIN 진료비
  ON 입원.입원ID = 진료비.입원ID
WHERE 퇴원일 IS NOT NULL;

In [ ]:
%%sql
-- [15번] 이렇게 고칩니다 → 70,299 (재원일수는 입원 표에서만)
SELECT SUM(퇴원일 - 입원일) AS 재원일수
FROM 입원
WHERE 퇴원일 IS NOT NULL;

In [ ]:
%%sql
-- [16번] 엑셀에서 앞자리 0 이 사라진 번호 → 0행
SELECT *
FROM 환자
WHERE 환자ID = '10019';

In [ ]:
%%sql
-- [17번] 환자번호는 글자 그대로 → 1행
SELECT *
FROM 환자
WHERE 환자ID = '0010019';

## 직접 해보기 — 6문항

`___` 를 채워 실행하면 ✅ / ❌ 가 나옵니다. 열 이름이나 값이 필요하면 바로 위 조회 셀을 먼저 실행하세요.

**Q1.** **환자 수** — 입원 표에서 환자는 몇 명? 중복 없이 셀 **열 이름**을 채우세요.

In [ ]:
%%sql
-- [18번] 입원 표의 열 이름 보기
SELECT *
FROM (DESCRIBE 입원);

In [ ]:
%%sql
-- [19번] Q1
SELECT COUNT(DISTINCT ___) AS 환자수
FROM 입원;

**Q2.** **2024년 퇴원 건수** — ‘퇴원 기준’으로 셉니다. 날짜 **열 이름**을 채우세요.

In [ ]:
%%sql
-- [20번] Q2
SELECT COUNT(*) AS 퇴원건수
FROM 입원
WHERE YEAR(___) = 2024;

**Q3.** **재원 중** — 12월 31일에 아직 재원 중인 입원은 몇 건? 빈 값을 찾는 말을 채우세요.

In [ ]:
%%sql
-- [21번] Q3
SELECT COUNT(*) AS 재원중
FROM 입원
WHERE 퇴원일 ___;

**Q4.** **진료과별 평균 재원일수** — 재원일수 = 퇴원일 − 입원일. 빼는 **열 이름**을 채우세요.

In [ ]:
%%sql
-- [22번] Q4
SELECT 진료과, ROUND(AVG(퇴원일 - ___), 1) AS 평균재원일수
FROM 입원
WHERE 퇴원일 IS NOT NULL
GROUP BY 진료과
ORDER BY 진료과;

**Q5.** **전체 병상 가동률(%)** — 2024년 날짜 수(윤년이라 366일)를 채우세요.

In [ ]:
%%sql
-- [23번] Q5
SELECT ROUND(100.0 * SUM(재원환자수)
             / ((SELECT SUM(병상수) FROM 병동) * ___), 1) AS 가동률
FROM 병동일보;

**Q6.** **비급여 금액** — 비급여 진료비의 합계는? 급여구분 **값**을 채우세요 (바로 위 셀에서 값 확인).

In [ ]:
%%sql
-- [24번] 급여구분에 어떤 값이 있는지 보기
SELECT DISTINCT 급여구분
FROM 진료비;

In [ ]:
%%sql
-- [25번] Q6
SELECT SUM(금액) AS 비급여금액
FROM 진료비
WHERE 급여구분 = ___;

## 오늘 배운 것

- 입원 건수와 환자 수는 다르다. 세는 단위와 날짜 기준을 먼저 정한다
- 재원일수 · 병상 가동률은 퇴원일과 병동일보로 구한다
- JOIN 전에 한 행의 단위를 확인한다. 1 : N 이면 합계가 부풀어난다

다음 시간: 13강. 의료 질 · 간호 지표 만들기

In [ ]:
진행확인()   # [26번]